# SEAWRD

Planetary interior modelling is a computationally demanding exercise, involving a lot of hydrodynamical considerations dependent on the composition and physical properties of an exoplanet. This takes a number of minutes, which grows to be incredibly large when performing hundreds of thousands of simulations.

A cheap approximation is availabe in the form of surrogate models. A neural network can act as a general function learner, i.e., something that maps inputs to outputs, and so we can use pre-ran expensive simulation data to train a small neural network to reproduce the simulation's results with great accuracy in a fraction of time. The hope is to train widely enough for this model to be used on new, never-seen exoplanet data and avoid expensive simulations.

This is what **S**urrogate **E**mulator for **A**quatic **W**orld **R**adius **D**etermination is for! Based on user-provided hyperparameters and data, it can train an appropriate surrogate model to be used in future research as an approximation for the full hydrodynamical simulations, namely as a predictor of the size of the planet.

## Imports

Note - in order to show off benchmarking functionality, not all imports are here; notably for `DNNManager`, `DNNTrainer`, and `Predictor`.

In [ ]:
import pandas as pd

from seawrd.preprocessing_data import DataPreprocessor
from seawrd.config_manager import ConfigManager
from seawrd.device_selection import choose_training_device
from seawrd.bootstrap import set_device_env

## Loading data

We are going to be using the simulation results from [Aguichine et al. (2021)](https://iopscience.iop.org/article/10.3847/1538-4357/abfa99), stored locally. This is from the result of expensive hydrodynamical simulations. The descriptions of each column is as follows:

* **x_core'** - the specific core mass fraction of the planet, equal to x_core / (1-x_H20)
    i.e., x_core' = 0.325 refers to an Earth-like CMF, regardles of the amount of water present
* **x_H20** - the water mass fraction of the planet
* **T_irr** - the irradiation temperature of the planet
* **T_b** - the boundary temperature i.e., at the outer boundary of the core.
* **M_b** - mass of the interior of the planet
* **M_a** - mass of the atmosphere of the planet
* **R_b** - radius of the interior of the planet
* **R_a** - radius of the atmosphere of the planet

In [ ]:
# Reading the data from the file
DATA_PATH = "DNN_data_IOP_Aguichine2021.dat"
data = pd.read_table(DATA_PATH, sep=r"\s+")

data.head(5)

The `DataPreprocessor` class is designed to parse Pandas dataframes like this into a format appropriate for model training, specifically validating and inferring feature & label columns & names, performing a training/test data split, and also creating a calibrated normaliser for use in the model architecture.

We need to specify what in the dataframe we are classing as features. In this case, we are deciding to simplify the model by combining the different masses and radii into one value for the planet, i.e., $M_p = M_b + M_a,  R_p = R_b + R_a$.

We also determine which column is the quality-control; in this datafile, if `errcode = 1` for any row, it means an error has occured within the simulation. We have chosen the simple method of removing any rows with errors in them.

We can also specify what fraction of the inputted data should be test data (`TEST_SIZE`) and whether to produce a normaliser, which we strongly recommend

In [ ]:
# Setting up the data to be processed properly
feature_names = ["x_core'", "x_H2O", "T_irr", "T_b", "M_p"]
label_name = "R_p"

# This is to do with errors in the data; any row with a 1 in the errcode column will not be included
quality_column = "errcode"
quality_threshold = 0

# Options for the DataPreprocessor
TEST_SIZE = 0.2
RANDOM_STATE = 42
NORMALISE = True
dp = DataPreprocessor(df=data,
                      features=feature_names,
                      label=label_name,
                      test_size=TEST_SIZE,
                      random_state=RANDOM_STATE,
                      quality_column=quality_column,
                      quality_value=quality_threshold,
                      normalise=NORMALISE)

# By default, this returns float32 NumPy arrays, which are favoured for training models. I have made this explicit
normaliser, train_features, test_features, train_labels, test_labels = dp.get_training_data(return_array=True)

print(f"Train features shape: {train_features.shape}")
print(train_features[:5])
print(f"\nTrain labels shape: {train_labels.shape}")
print(train_labels[:5])
print(f"\nTest features shape: {test_features.shape}")
print(test_features[:5])
print(f"\nTest labels shape: {test_labels.shape}")
print(test_labels[:5])

## Program config

This project makes use of a TOML configuration file to control important parts of the program without needing to change the code. The default config is found at `seawrd_default.toml` and configs can be loaded from files using the `ConfigManager`.

In [ ]:
cfg_manager = ConfigManager.from_toml("seawrd_default.toml")
config = cfg_manager.config

The highest-level config option is the `SEAWRDConfig` class, which, along with some simple methods, contains a bunch of different derived data classes, of base `ConfigSection` - e.g., `ModelConfig`, one for each section in the TOML file. Each one of these data classes has variables corresponding to different configurable options, e.g., `ModelConfig.num_neurons`.

The advantage of this approach, beyond simple encapsulation and possible future customisation, is that validation of every single configuration value now happens automatically when a `ConfigSection` is loaded or created. Not super important for you to know but it's easy to use anyway! All config objects are found in config.py.

In [ ]:
print(type(config))
print(type(config.model))
print(config.model.num_neurons)

Alternatively, it is also possible to create a full `SEAWRDConfig`, a single `ConfigSection`, or any smaller part (i.e., a single field or above) using dictionaries. All objects just mentioned have a `from_dict` method and can also be updated with the `with_update` method for `ConfigSection` objects or `with_overrides` mmethod for `ConfigManager`.

## Selecting a training device

SEAWRD can automatically decide whether to train on CPU or GPU by briefly benchmarking both on your actual model architecture and data, then picking whichever is faster. This happens automatically when you use the `seawrd-train` command-line entrypoint (see `seawrd.cli.train`), and is also available directly via `seawrd.device_selection.choose_training_device`.

The `[device]` section of the config controls this:
* **mode** - `"auto"` benchmarks and picks the faster device; `"cpu"`/`"gpu"` forces that device without benchmarking.
* **min_gpu_speedup** - in `"auto"` mode, the GPU is only chosen if it is at least this many times faster than the CPU.
* **benchmark_epochs**, **warmup_epochs**, **benchmark_repeats** - control how long/thorough the benchmark is.

Each device is benchmarked in its own isolated subprocess, so a GPU-related crash can't take down your whole training run, and results are cached by default so repeated runs with the same config and data shape skip re-benchmarking. We'll use a much shorter benchmark than the config default here just to keep this demo fast.

In [ ]:
# Benchmark CPU vs GPU on our actual data and pick the faster one (using a short benchmark for this demo)
device_choice = choose_training_device(
    config=config.to_dict(),
    x_train=train_features,
    y_train=train_labels,
    x_val=test_features,
    y_val=test_labels,
    benchmark_epochs=5,
    benchmark_repeats=1,
    warmup_epochs=2,
)

print(f"Selected device: {device_choice.device}")
print(f"Reason: {device_choice.reason}")

# Apply the choice by setting GPU visibility before building/training the model below
set_device_env(device_choice.device)

## Create a DNN Model manager

The class `DNNManager` is used to create and manage dense neural network (DNN) models. In the normal processing of the code, it is largely in the back-end, with a separate class `DNNTrainer` using an instance for training purposes.

A DNN model has several different layers of neurons. The first layer is an input layer, followed by a normalisation layer, which will take a row of training features (i.e., one value for every feature). There is then a number of hidden layers, followed by an output layer, which will produce a label(s) (in this case, we want to predict the radius of the planet $R_p$)

We have to decide some important hyperparameters of the model in the config file:
- **num_hidden_layers** - the number of hidden layers
- **num_neurons** - the number of neurons in each hidden layer

While adding more layers and more neurons is likely to increase model performance (i.e., make it more accurate) up to some point, it will also greatly increase training time. It is up to you to balance the architecture!

In [ ]:
from seawrd.model import DNNManager

# Generate the model
dnn_manager = DNNManager.from_config(
    model_config=config.model,
    input_shape=train_features.shape[1:],
    normaliser=normaliser
)
dnn_model = dnn_manager.model
dnn_model.summary()

Every model has a `model_name`, which describes the architecture above.

'R' stands for Rectangular, which is the type of basic network represented here. Then you have `num_layers`x`num_neurons` to show how many hidden layers of how many neurons are included. Then you have `num_inputs`i and `num_outputs`o.

In [ ]:
print(dnn_manager.model_name)

It is also possible to save and load a model into the class, allowing for further training and the usage of previously-created models. This is available through the `DNNManager.from_previous_model()` constructor and the `save_model_version()` method.

`save_model_version()` (and, as we'll use below, `DNNTrainer.train_models()`) also accepts optional `feature_names` and `label_name` arguments. When given, a small JSON *manifest* recording them is saved alongside the model — this is what lets the `Predictor` class (see the **Making Predictions** section at the end of this notebook) reload a model later and safely match up new data to it, without you needing to remember or hard-code the exact feature order it was trained on.

For more informations on Dense Neural Networks, see [here](https://www.scribd.com/document/480813741/AML-03-Dense-Neural-Networks).

## Model Training

A model is initialised with random small weights and biases, and so will initially be very poorly performing. During training, the model's weights and biases are updated in order to minimise or maximise a given loss function, depending on your preference.

The `DNNTrainer` class is designed to train a model architecture to be able to predict the planetary radius given a feature vector by training a number of different randomly initialised models and choosing the best one. This is because when using small neural networks, the random initialisation will actually affect the final performance of the model.

This requires a `DNNManager` instance, like the one we created before. We must also specify how many epochs (`NUM_EPOCHS`) to train each model for (more on that later!) and the version the final model will be called. Note, we have set `NUM_EPOCHS` to a very small value here in order to demonstrate functionality, you will most definitely want to run it for more epochs.

There are also other hyperparameters:
* **learning_rate** - a very important hyperparameter, essentially how large a change should be made in the model's weights and biases during each training step. This is decreased during training to smoothly approach minima.
* **batch_size** - how many feature vectors to include in a single training step; the model is evaluated based on the mean-squared error of all of these predictions, rather than just a single one.
* **validation_split** - during training, the performance of the model is evaluated by testing its predictions on validation data. Validation data is not used by the model in training, and is split from the inputted training data, based on this fraction. 

In [ ]:
from seawrd.trainer import DNNTrainer

# Overwrite the config for this example
config = cfg_manager.with_override("training.num_epochs", 200).config

dnn_trainer = DNNTrainer(model_manager=dnn_manager, config=config)

Because our DNNs are small in size, the final result depends heavily on our initial (random) state. To combat this, we will train the same model architecture multiple (`NUM_MODELS`) times, each one having different random initial conditions (the `seed`). While training, we will keep track of statistics about each model's performance and save only the best performing model in the end.

During training, extra actions are also taken! If `output.save_model` is enabled (the default), the best-performing model is saved to disk. Below we also pass along the feature and label names that `DataPreprocessor` inferred for us (`dp.feature_names_` and `dp.label_name_`), so that a prediction manifest is saved alongside the model too.

In [ ]:
# Train the models
rp_means, rp_stds, losses, val_losses = dnn_trainer.train_models(input_features=train_features,
                                                                 input_labels=train_labels,
                                                                 test_features=test_features,
                                                                 test_labels=test_labels,
                                                                 feature_names=dp.feature_names_,
                                                                 label_name=dp.label_name_)

With the model finished training, we can see some statistics!

In [ ]:
# Print the architecture performance of the best model
dnn_trainer.print_architecture_performance()

We can also visualise the loss curve of the best model from training, compared against the validation loss this is not going to be great for our very short example but you'll be able to clearly see improvements w/ epochs!

In [ ]:
dnn_trainer.plot_loss_curve(log_y=True, log_x=False, max_y=1)

## Making Predictions

Now that we have a trained (and saved) model, along with a manifest of the feature names it expects, we can use the `Predictor` class to run inference on new data — without needing to separately track or hard-code the feature order the model was trained on.

`Predictor.from_saved()` loads both the saved model and its manifest (if one exists). When the manifest is present, predicting on a `pandas.DataFrame` automatically selects and reorders its columns to match the training order, and raises a clear error if any required feature is missing.

In [ ]:
from seawrd.predictor import Predictor

predictor = Predictor.from_saved(
    model_dir=dnn_trainer.model_dir,
    model_name=dnn_trainer.model_name,
    version=dnn_trainer.version,
)

print("Feature order the model expects:", predictor.feature_names)
print("Label the model predicts:", predictor.label_name)

For example, here's a small sample of new planetary data with its columns deliberately shuffled relative to the training order — `Predictor` still lines them up correctly:

In [ ]:
# Columns are deliberately out of training order here
sample = pd.DataFrame({
    "T_b": [400.0, 300.0],
    "M_p": [3.0, 2.0],
    "x_H2O": [0.3, 0.4],
    "T_irr": [350.0, 250.0],
    "x_core'": [0.3, 0.2],
})

predictor.predict_dataframe(sample)

This same workflow is available directly from the command line once a model has been trained and saved, via the `seawrd-predict` entrypoint:

```bash
seawrd-predict new_data.dat --model-name <model_name> --model-dir models/
```

which reads a whitespace-delimited table of features, aligns them using the saved manifest, and writes (or prints) the predictions — see `seawrd.cli.predict` for details.